In [ ]:
import os
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import cv2
import numpy as np
import qrcode
from PIL import Image

In [ ]:
DATASET_DIR = "dataset"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_DIR = os.path.join(DATASET_DIR, "labels")

QR_SIZE = 21

DATASET_SIZE = 20_000
IMAGE_SIZE = QR_SIZE * 10

START_DATETIME = datetime(2023, 1, 1, 0, 0, 0)
END_DATETIME = datetime(2026, 12, 31, 23, 59, 59)

In [3]:
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(LABELS_DIR, exist_ok=True)

In [4]:
def get_random_datetime(start: datetime, end: datetime) -> datetime:
    total_seconds = int((end - start).total_seconds())
    return start + timedelta(seconds=random.randint(0, total_seconds))


def build_qr_code(payload: str) -> Tuple[List[List[bool]], np.ndarray]:
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_M,
        box_size=1,
        border=0,
    )
    qr.add_data(payload)
    qr.make(fit=False)

    matrix = qr.get_matrix()
    base = Image.new("1", (QR_SIZE, QR_SIZE), 1)
    pixel_access = base.load()
    
    for y in range(QR_SIZE):
        for x in range(QR_SIZE):
            pixel_access[x, y] = 0 if matrix[y][x] else 1

    image = base.resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST).convert("L")
    return matrix, np.array(image)

In [5]:
def apply_perspective(image: np.ndarray) -> np.ndarray:
    height, width = image.shape

    margin = 0.05 * width
    src = np.float32([[0, 0], [width, 0], [width, height], [0, height]])
    dst = np.float32([
        [random.uniform(0, margin), random.uniform(0, margin)],
        [width - random.uniform(0, margin), random.uniform(0, margin)],
        [width - random.uniform(0, margin), height - random.uniform(0, margin)],
        [random.uniform(0, margin), height - random.uniform(0, margin)],
    ])

    matrix = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(image, matrix, (width, height), borderValue=255)


def apply_anisotropic_scale(image: np.ndarray) -> np.ndarray:
    scale_x = random.uniform(0.95, 1.05)
    scale_y = random.uniform(0.95, 1.05)

    image = cv2.resize(
        image,
        None,
        fx=scale_x,
        fy=scale_y,
        interpolation=cv2.INTER_NEAREST,
    )

    return cv2.resize(
        image,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=cv2.INTER_NEAREST,
    )


def apply_blur(image: np.ndarray) -> np.ndarray:
    if random.random() < 0.5:
        return cv2.GaussianBlur(image, (3, 3), 0)
    return image


def apply_contrast(image: np.ndarray) -> np.ndarray:
    alpha = random.uniform(0.8, 1.2)
    beta = random.uniform(-15, 15)
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)


def augment_image(image: np.ndarray) -> np.ndarray:
    image = apply_perspective(image)
    image = apply_anisotropic_scale(image)
    image = apply_blur(image)
    image = apply_contrast(image)
    return image

In [6]:
def write_label(path: str, matrix: List[List[bool]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in matrix:
            f.write(" ".join("1" if v else "0" for v in row) + "\n")

In [7]:
used_names = set()
for i in range(DATASET_SIZE):
    while True:
        datetime = get_random_datetime(START_DATETIME, END_DATETIME)
        basename = datetime.strftime("%d%m%Y%H%M%S")
        if basename not in used_names:
            used_names.add(basename)
            break

    payload = datetime.strftime("%d-%m-%Y %H:%M:%S")

    matrix, qr_code = build_qr_code(payload)
    augmented_qr_code = augment_image(qr_code)

    image_path = os.path.join(IMAGES_DIR, f"{basename}.png")
    label_path = os.path.join(LABELS_DIR, f"{basename}.txt")
    
    cv2.imwrite(image_path, augmented_qr_code)
    write_label(label_path, matrix)

    if (i+1) % 500 == 0:
        print(f"{i+1}/{DATASET_SIZE} images generated.")

500/20000 images generated.
1000/20000 images generated.
1500/20000 images generated.
2000/20000 images generated.
2500/20000 images generated.
3000/20000 images generated.
3500/20000 images generated.
4000/20000 images generated.
4500/20000 images generated.
5000/20000 images generated.
5500/20000 images generated.
6000/20000 images generated.
6500/20000 images generated.
7000/20000 images generated.
7500/20000 images generated.
8000/20000 images generated.
8500/20000 images generated.
9000/20000 images generated.
9500/20000 images generated.
10000/20000 images generated.
10500/20000 images generated.
11000/20000 images generated.
11500/20000 images generated.
12000/20000 images generated.
12500/20000 images generated.
13000/20000 images generated.
13500/20000 images generated.
14000/20000 images generated.
14500/20000 images generated.
15000/20000 images generated.
15500/20000 images generated.
16000/20000 images generated.
16500/20000 images generated.
17000/20000 images generated.
